In [82]:
import collections
import json
import pickle
import re

import pandas as pd

# Configuration
TABOO_FLAG = 'CURRENT'  # match combined.ipynb behaviour

DECKLIST_JSON_PICKLE = 'decklist_json.pickle'
CARD_JSON_PICKLE = 'card_json.pickle'
TABOO_FILE = 'taboo.json'

# Load previously-scraped decklists and cards from disk
with open(DECKLIST_JSON_PICKLE, 'rb') as f:
  decklist_json = pickle.loads(f.read())
with open(CARD_JSON_PICKLE, 'rb') as f:
  card_json = pickle.loads(f.read())

print(f"Loaded {len(decklist_json)} raw decklists")
print(f"Loaded {len(card_json)} cards")

Loaded 63171 raw decklists
Loaded 2046 cards


In [83]:
# Basic decklist cleaning: drop empty entries and joke decks

# Drop None / empty decklists
decklist_json = {decklist_id: deck
                  for decklist_id, deck in decklist_json.items()
                  if deck}

# Remove known joke decklists (copied from combined.ipynb)
for bad_id in (44599, 43839):
  if bad_id in decklist_json:
    decklist_json.pop(bad_id)

print(f"{len(decklist_json)} decklists after removing empty/joke entries")

55731 decklists after removing empty/joke entries


In [84]:
# Load taboo data and helpers (adapted from combined.ipynb)

with open(TABOO_FILE, 'r', encoding='utf-8') as f:
  taboo_json = json.load(f)

# Find id of most recent taboo
MAX_TABOO = max([taboo_dict['id'] for taboo_dict in taboo_json])
print(f"Most recent taboo id: {MAX_TABOO}")

# Build lookup: taboo_id -> {card_code -> taboo_card}
taboo_dict = collections.defaultdict(dict)
for taboo in taboo_json:
  taboo_id = taboo['id']
  for taboo_card in json.loads(taboo['cards']):
    card_code = taboo_card['code']
    taboo_dict[taboo_id][card_code] = taboo_card

  if taboo_id == MAX_TABOO:
    # list of card_codes in most recent taboo that change xp or are forbidden
    tabooed_cards = [card['code'] for card in json.loads(taboo['cards'])
                     if 'xp' in card or card['text'] == 'Forbidden.']


def lookup_taboo(card_code: str, taboo_id: int | None):
  """Return taboo card entry for (card_code, taboo_id), or None if missing."""
  if taboo_id in taboo_dict:
    return taboo_dict[taboo_id].get(card_code, None)
  return None


def lookup_taboo_xp(card_code: str, taboo_id: int | None) -> int:
  """Return taboo xp modifier for (card_code, taboo_id), defaulting to 0."""
  taboo_card = lookup_taboo(card_code, taboo_id)
  if taboo_card is not None:
    return taboo_card.get('xp', 0)
  return 0

Most recent taboo id: 10


In [85]:
# Map card_id -> canonical_id (spec.md fingerprint + duplicate_of_code)
# and rewrite decklist slots / taboo keys to canonical_id.

import importlib
import arkham_canonical
importlib.reload(arkham_canonical)
from arkham_canonical import CanonicalMapper, parse_investigator_front_back

canonical_mapper = CanonicalMapper(card_json, chapter=1)
canonical_id_map = canonical_mapper.canonical_id_map
canonical_cycle = canonical_mapper.canonical_cycle


def to_canonical(card_id: str) -> str:
  """Return canonical_id for a card_id (identity if unmapped)."""
  return canonical_mapper.to_canonical(card_id)


def to_canonical_front(card_id: str) -> str:
  """Return canonical_front for an investigator front card_id."""
  return canonical_mapper.to_canonical_front(card_id)


def to_canonical_back(card_id: str) -> str:
  """Return canonical_back for an investigator back card_id."""
  return canonical_mapper.to_canonical_back(card_id)


def decklist_canonical_front_back(decklist: dict) -> tuple[str, str]:
  """Return (canonical_front, canonical_back) for a decklist."""
  return canonical_mapper.decklist_canonical_front_back(decklist)


def cycle_for_slot(card_id: str) -> int | None:
  """Return CanonicalCard.cycle for a slot code, or None if out-of-order or missing."""
  return canonical_mapper.cycle_for_slot(card_id)


def decklist_cycle(slots: dict) -> int | None:
  """Return Decklist.cycle: max cycle among known slot codes."""
  return canonical_mapper.decklist_cycle(slots)


def decklist_has_unknown_slots(slots: dict) -> bool:
  """Return True if any slot code is missing from card_json."""
  return canonical_mapper.decklist_has_unknown_slots(slots)


# Merge decklist card codes
for decklist in decklist_json.values():
  slots = decklist['slots']
  for card_code in list(slots.keys()):
    canonical_id = to_canonical(card_code)
    if canonical_id == card_code:
      continue

    num = slots.pop(card_code)
    slots[canonical_id] = slots.get(canonical_id, 0) + num


# Merge taboo card codes so taboo lookups stay consistent
for taboo_id in sorted(taboo_dict.keys()):
  taboo = taboo_dict[taboo_id]
  for card_code, taboo_card in list(taboo.items()):
    canonical_id = to_canonical(card_code)
    if canonical_id == card_code:
      continue
    taboo_dict[taboo_id][canonical_id] = taboo_card

# tabooed_cards is used for CURRENT-taboo xp checks on canonical slot codes
tabooed_cards = sorted({to_canonical(card_code) for card_code in tabooed_cards})

reprint_card_ids = sum(1 for card_id, canonical_id in canonical_id_map.items()
                       if card_id != canonical_id)
print(f"Mapped {len(card_json)} card_ids -> {len(set(canonical_id_map.values()))} canonical_ids")
print(f"Merged {reprint_card_ids} reprint card_ids in decklists and taboo data")

Mapped 2046 card_ids -> 1653 canonical_ids
Merged 143 reprint card_ids in decklists and taboo data


In [86]:
# Cycle stats (spec Pack Order; uses CanonicalCard.cycle from arkham_canonical)

cycle_card_count = collections.Counter()
for card_code, card in card_json.items():
  # Ignore investigator-specific signature cards
  if 'restrictions' in card and 'investigator' in card['restrictions']:
    continue
  # Ignore basic weaknesses
  if card.get('subtype_code') == 'basicweakness':
    continue
  # Count each canonical_id once
  if to_canonical(card_code) != card_code:
    continue

  cycle = cycle_for_slot(card_code)
  if cycle is None:
    continue
  cycle_card_count[cycle] += 1

cycle_card_cumulative: dict[int, int] = {}
so_far = 0
for cycle in sorted(cycle_card_count):
  so_far += cycle_card_count[cycle]
  cycle_card_cumulative[cycle] = so_far

deck_cycle_by_card_cycle: dict[int, dict[int, int]] = collections.defaultdict(
  lambda: collections.defaultdict(int)
)
unknown_slot_refs = 0
for decklist in decklist_json.values():
  slots = decklist.get('slots') or {}
  deck_cycle = decklist_cycle(slots)
  if deck_cycle is None:
    continue

  for card_code, num in slots.items():
    if not canonical_mapper.is_known_card(card_code):
      unknown_slot_refs += num
      continue
    card_cycle = cycle_for_slot(card_code)
    if card_cycle is None:
      continue
    deck_cycle_by_card_cycle[deck_cycle][card_cycle] += num

cycle_pivot = pd.DataFrame.from_dict(
  {deck_cycle: dict(card_counts) for deck_cycle, card_counts in deck_cycle_by_card_cycle.items()},
  orient='index',
).fillna(0).astype(int)
cycle_pivot.index.name = 'Decklist.cycle'
cycle_pivot.columns.name = 'CanonicalCard.cycle'
cycle_pivot = cycle_pivot.sort_index().reindex(sorted(cycle_pivot.columns), axis=1)

cycle_pivot['All'] = cycle_pivot.sum(axis=1)
all_row = cycle_pivot.drop(columns='All').sum(axis=0)
all_row['All'] = int(cycle_pivot.drop(columns='All').values.sum())
cycle_pivot.loc['All'] = all_row

print("Cycle card counts:")
print(dict(sorted(cycle_card_count.items())))
print("Cumulative cards by cycle:")
print(cycle_card_cumulative)
print("Slot copies by Decklist.cycle x CanonicalCard.cycle (margins reproduce prior totals):")
display(cycle_pivot)
if unknown_slot_refs:
  print(f"Skipped {unknown_slot_refs} slot copies with unknown card codes")

Cycle card counts:
{1: 86, 2: 107, 3: 112, 4: 114, 5: 105, 6: 96, 7: 122, 8: 116, 9: 139, 10: 114, 11: 119, 12: 126, 13: 215}
Cumulative cards by cycle:
{1: 86, 2: 193, 3: 305, 4: 419, 5: 524, 6: 620, 7: 742, 8: 858, 9: 997, 10: 1111, 11: 1230, 12: 1356, 13: 1571}
Slot copies by Decklist.cycle x CanonicalCard.cycle (margins reproduce prior totals):


CanonicalCard.cycle,1,2,3,4,5,6,7,8,9,10,11,12,13,All
Decklist.cycle,,,,,,,,,,,,,,
1,74771,0,0,0,0,0,0,0,0,0,0,0,0,74771
2,141653,57272,0,0,0,0,0,0,0,0,0,0,0,198925
3,103199,41415,35142,0,0,0,0,0,0,0,0,0,0,179756
4,80831,35052,33287,27512,0,0,0,0,0,0,0,0,0,176682
5,98977,41768,37659,31285,44515,0,0,0,0,0,0,0,0,254204
6,50725,23060,22408,20073,24563,19678,0,0,0,0,0,0,0,160507
7,65499,24976,17981,16459,14573,9331,42663,0,0,0,0,0,0,191482
8,33905,14429,11859,10983,13341,10959,12109,16766,0,0,0,0,0,124351
9,61881,21764,21062,15310,15843,12030,18455,9164,27041,0,0,0,0,202550


Skipped 1 slot copies with unknown card codes


In [73]:
# Normalize by row
cycle_pivot.div(cycle_pivot.sum(axis=1) / 2, axis=0)

CanonicalCard.cycle,1,2,3,4,5,6,7,8,9,10,11,12,13,All
Decklist.cycle,,,,,,,,,,,,,,
1,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
2,0.712092,0.287908,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
3,0.574106,0.230396,0.195498,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
4,0.457494,0.198390,0.188401,0.155715,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
5,0.389361,0.164309,0.148145,0.123070,0.175115,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
6,0.316030,0.143670,0.139608,0.125060,0.153034,0.122599,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
7,0.342063,0.130435,0.093904,0.085956,0.076106,0.048730,0.222804,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
8,0.272656,0.116034,0.095367,0.088323,0.107285,0.088130,0.097378,0.134828,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
9,0.305510,0.107450,0.103984,0.075586,0.078218,0.059393,0.091113,0.045243,0.133503,0.000000,0.000000,0.000000,0.000000,1.0


In [87]:
# XP and investigator name helpers (adapted from combined.ipynb)

UPGRADE_PATTERN = re.compile(r"\d+\|\d+.*")


def parse_customizable(customizable_string: str):
  """Parse ArkhamDB customizable upgrade string.

  Returns (indices_list, xp_list) where each index is an upgrade index and
  each xp is its cost, filtered to upgrades with positive xp.
  """
  upgrade_list = customizable_string.split(',')
  upgrade_list = [u.split('|') for u in upgrade_list
                  if UPGRADE_PATTERN.match(u)]
  indices_list = [u[0] for u in upgrade_list if int(u[1]) > 0]
  xp_list = [int(u[1]) for u in upgrade_list if int(u[1]) > 0]
  return indices_list, xp_list


def calc_card_code_xp(card_code: str, taboo_id: int | None) -> int:
  card_dict = card_json[card_code]
  card_xp = card_dict.get('xp', 0)
  exceptional = card_dict.get('exceptional', False)

  # Apply taboo modifications if we are pinning to the latest taboo
  if TABOO_FLAG == 'CURRENT' and card_code in tabooed_cards:
    card_taboo = lookup_taboo(card_code, MAX_TABOO)
    if card_taboo is not None:
      if 'xp' in card_taboo:
        card_xp += card_taboo['xp']
      if 'exceptional' in card_taboo:
        exceptional = card_taboo['exceptional']

  if exceptional:
    card_xp *= 2

  return card_xp


def decklist_investigator_name(decklist: dict) -> str:
  """Return investigator name, treating parallel fronts/backs as distinct."""
  investigator_name = decklist['investigator_name']
  return investigator_name


def decklist_investigator_front_back(decklist: dict) -> tuple[str, str]:
  """Return raw investigator_front/back card_ids from a decklist."""
  return parse_investigator_front_back(decklist)


def decklist_xp(decklist: dict) -> int:
  """Return total XP of the decklist, including customizable upgrades."""
  xp = 0
  taboo_id = decklist.get('taboo_id')

  for card_code, num in decklist['slots'].items():
    card_dict = card_json[card_code]
    card_xp = calc_card_code_xp(card_code, taboo_id)

    if card_dict.setdefault('myriad', False):
      xp += card_xp
    else:
      xp += card_xp * num

  meta = decklist.get('meta', '')
  if meta:
    customizable_cards = json.loads(meta)
    for key, customizable_string in customizable_cards.items():
      if key.startswith('cus_'):
        _, xp_list = parse_customizable(customizable_string)
        xp += sum(xp_list)

  # Global XP adjustments
  if '08125' in decklist['slots']:
    xp = max(0, xp - 3)  # In the Thick of It

  if decklist_investigator_name(decklist) == 'Kymani Jones':
    xp = max(0, xp - 5)

  return xp

In [88]:
# Build decklist DataFrame via spec.md pipeline (D1–D4)
# Tolerates missing card_json entries until cards are re-scraped.

import importlib
import arkham_popularity
importlib.reload(arkham_popularity)
from arkham_popularity import ArkhamPopularityEngine, prepared_decks_to_dataframe

pop_engine = ArkhamPopularityEngine(card_json, canonical_mapper, taboo_json)
prepared_decks = pop_engine.prepare_all(decklist_json)
df_all_decks = prepared_decks_to_dataframe(prepared_decks)

print(df_all_decks.shape)
print(
  f"Ignoring {df_all_decks['is_ignore'].sum()} decks; "
  f"keeping {(~df_all_decks['is_ignore']).sum()}"
)
print(
  'Unknown slots:',
  int(df_all_decks['has_unknown_slots'].sum()),
  '| Chapter 2:',
  int(df_all_decks['has_chapter_2_cards'].sum()),
)
print(df_all_decks[[
  'id', 'investigator_name',
  'canonical_front', 'canonical_back',
  'cycle', 'xp_cost', 'is_ignore',
]].head())

(55731, 20)
Ignoring 11210 decks; keeping 44521
Unknown slots: 1 | Chapter 2: 518
      id investigator_name canonical_front canonical_back  cycle  xp_cost  \
0  25345       Agnes Baker           01004          01004    5.0        0   
1  25344       Wendy Adams           01005          01005    4.0        0   
2  25343      Stella Clark           60501          60501    7.0        0   
3  25342       Agnes Baker           01004          01004    2.0        0   
4  25341      Roland Banks           01001          01001    1.0        5   

   is_ignore  
0      False  
1       True  
2      False  
3      False  
4      False  


In [89]:
# is_ignore was set by ArkhamPopularityEngine (D4: taboo_set, unknown slots,
# Chapter 2). Keep non-ignored decks for analysis.

df_decks = df_all_decks[~df_all_decks['is_ignore']].copy()
print(f"{len(df_decks)} non-ignored decks")

44521 non-ignored decks


In [90]:
# Spec Y1/Y2: user_weight and Cycle.weight

user_weight_by_id = pop_engine.assign_user_weights(prepared_decks)
df_decks['user_weight'] = df_decks['id'].map(user_weight_by_id)

cycle_weight_by_cycle = pop_engine.assign_cycle_weights(
  prepared_decks,
  user_weight_by_id,
)
df_decks['cycle_weight'] = df_decks['cycle'].map(cycle_weight_by_cycle)

df_decks['deck_weight'] = df_decks['user_weight'] * df_decks['cycle_weight']

print(f"Total user_weight: {df_decks['user_weight'].sum():.1f}")
print(f"Total deck_weight: {df_decks['deck_weight'].sum():.4f}")
print('Cycle weights:', {k: round(v, 6) for k, v in sorted(cycle_weight_by_cycle.items())})

# Sanity check (spec Y2): deck counts and sum_user_weight per cycle
cycle_sanity = (
  df_decks[df_decks['cycle'].notna()]
  .groupby('cycle', as_index=False)
  .agg(deck_count=('id', 'count'), sum_user_weight=('user_weight', 'sum'))
  .sort_values('cycle')
)
cycle_sanity['sum_user_weight'] = cycle_sanity['sum_user_weight'].round(1)
cycle_sanity['cycle_weight'] = (1.0 / cycle_sanity['sum_user_weight']).round(6)
print('\nDecks and sum_user_weight by cycle (non-ignored, cycle defined):')
display(cycle_sanity)

none_decks = df_decks[df_decks['cycle'].isna()]
if len(none_decks):
  print(
    f"cycle=None: {len(none_decks)} decks, "
    f"sum_user_weight={none_decks['user_weight'].sum():.1f}"
  )

Total user_weight: 21253.2
Total deck_weight: 10.2543
Cycle weights: {1: 0.000425, 2: 0.000425, 3: 0.000425, 4: 0.000425, 5: 0.000425, 6: 0.000443, 7: 0.000443, 8: 0.000443, 9: 0.000443, 10: 0.000523, 11: 0.000593, 12: 0.001107}

Decks and sum_user_weight by cycle (non-ignored, cycle defined):


,cycle,deck_count,sum_user_weight,cycle_weight
0,1.0,1880,1054.9,0.000948
1,2.0,4549,2089.5,0.000479
2,3.0,4233,1981.5,0.000505
3,4.0,4054,1827.7,0.000547
4,5.0,5538,2352.8,0.000425
5,6.0,3366,1530.5,0.000653
6,7.0,4445,2126.7,0.000470
7,8.0,3184,1529.9,0.000654
8,9.0,4504,2256.4,0.000443
9,10.0,3794,1913.5,0.000523


cycle=None: 1 decks, sum_user_weight=1.0


In [91]:
# deck_weight is the spec popularity weight (user_weight * Cycle.weight).
# Keep group_score as an alias for downstream cells until they are migrated.
df_decks['group_score'] = df_decks['deck_weight']
print(f"Total group_score (deck_weight): {df_decks['group_score'].sum():.4f}")

Total group_score (deck_weight): 10.2543


In [92]:
# Helper used throughout: adjust scores for uneven pack / cycle availability


def group_and_score(df: pd.DataFrame,
                    groupby_cols: list[str],
                    score_col: str,
                    pack_index_col: str):
  """Aggregate scores per group, normalized by pack availability.

  This is a direct analogue of the function in prepare_data.ipynb,
  specialized for Arkham where `pack_index_col` is publication cycle.
  """
  # For each (group, pack), total score
  df_group_pack = (
    df.groupby(groupby_cols + [pack_index_col])[score_col]
      .sum()
      .reset_index()
  )

  # For each pack, average score per group
  df_pack_map = (
    df_group_pack.groupby(pack_index_col)[score_col]
      .mean()
      .to_dict()
  )

  # Attach pack-wise average expectation
  df_group_pack['pack_avg'] = df_group_pack[pack_index_col].map(df_pack_map)

  # For each group, total score and total expected score
  df_group = (
    df_group_pack.groupby(groupby_cols)[[score_col, 'pack_avg']]
      .sum()
      .reset_index()
  )

  df_group['score'] = df_group[score_col] / df_group['pack_avg']
  return df_group.sort_values('score', ascending=False)

In [93]:
# Investigator popularity (spec I1–I5): compare investigators within each inv_cycle.
# Pooling all inv_cycles in one table overweights late-cycle investigators because
# Decklist.cycle >= 12 decks are weighted heavily under g(C) = C.

inv_name_map = (
  df_decks[['canonical_front', 'canonical_back', 'investigator_name']]
  .drop_duplicates(['canonical_front', 'canonical_back'])
  .set_index(['canonical_front', 'canonical_back'])['investigator_name']
  .to_dict()
)

inv_pop_rows = pop_engine.investigator_popularity_by_cycle(prepared_decks)
investigator_popularity = pd.DataFrame(inv_pop_rows)
investigator_popularity['investigator_name'] = investigator_popularity.apply(
  lambda row: inv_name_map.get(
    (row['canonical_front'], row['canonical_back']),
    row['canonical_front'],
  ),
  axis=1,
)
investigator_popularity = investigator_popularity.rename(columns={
  'inv_cycle': 'Cycle',
  'popularity': 'Popularity',
})

display_cols = [
  'investigator_name', 'canonical_front', 'canonical_back',
  'Cycle', 'Popularity', 'investigator_weight', 'pool_weight',
]

pd.set_option('display.max_rows', None)
for inv_cycle in sorted(investigator_popularity['Cycle'].dropna().unique()):
  print(f'=== inv_cycle = {int(inv_cycle)} (pool: Decklist.cycle >= {int(inv_cycle)}) ===')
  subset = (
    investigator_popularity[investigator_popularity['Cycle'] == inv_cycle]
    .sort_values('Popularity', ascending=False)
  )
  display(subset[display_cols])

=== inv_cycle = 1 (pool: Decklist.cycle >= 1) ===


,investigator_name,canonical_front,canonical_back,Cycle,Popularity,investigator_weight,pool_weight
12,Roland Banks,01001,01001,1,0.050513,0.517976,10.254349
4,Daisy Walker,01002,01002,1,0.043866,0.449816,10.254349
3,Agnes Baker,01004,01004,1,0.034641,0.355223,10.254349
2,"""Skids"" O'Toole",01003,01003,1,0.025523,0.261725,10.254349
5,Wendy Adams,01005,01005,1,0.018704,0.191794,10.254349
11,Roland Banks,01001,90024,1,0.000551,0.005647,10.254349
7,"""Skids"" O'Toole",01003,90008,1,0.000458,0.004696,10.254349
9,Daisy Walker,01002,90001,1,0.000443,0.004548,10.254349
6,Agnes Baker,01004,90017,1,0.000216,0.002216,10.254349
10,Wendy Adams,01005,90037,1,0.000152,0.001559,10.254349


=== inv_cycle = 2 (pool: Decklist.cycle >= 2) ===


,investigator_name,canonical_front,canonical_back,Cycle,Popularity,investigator_weight,pool_weight
13,Zoey Samaras,02001,02001,2,0.054699,0.536378,9.805978
19,Jenny Barnes,02003,02003,2,0.040465,0.396799,9.805978
17,Rex Murphy,02002,02002,2,0.028164,0.276171,9.805978
21,Jim Culver,02004,02004,2,0.021350,0.209354,9.805978
20,"""Ashcan"" Pete",02005,02005,2,0.020996,0.205884,9.805978
15,Zoey Samaras,02001,90059,2,0.000113,0.001107,9.805978
16,Jim Culver,02004,90049,2,0.000106,0.001036,9.805978
18,Jenny Barnes,02003,90084,2,0.000060,0.000593,9.805978
14,"""Ashcan"" Pete",02005,90046,2,0.000053,0.000523,9.805978


=== inv_cycle = 3 (pool: Decklist.cycle >= 3) ===


...

,investigator_name,canonical_front,canonical_back,Cycle,Popularity,investigator_weight,pool_weight
60,Charlie Kane,09018,09018,10,0.035021,0.105062,3.0
62,Kymani Jones,09008,09008,10,0.020698,0.062094,3.0
63,Vincent Lee,09004,09004,10,0.019394,0.058182,3.0
61,Carson Sinclair,09001,09001,10,0.017059,0.051178,3.0
59,Amina Zidane,09011,09011,10,0.008934,0.026803,3.0
64,Darrell Simmons,09015,09015,10,0.008695,0.026086,3.0


=== inv_cycle = 11 (pool: Decklist.cycle >= 11) ===


,investigator_name,canonical_front,canonical_back,Cycle,Popularity,investigator_weight,pool_weight
66,Alessandra Zorzi,10009,10009,11,0.032530,0.065060,2.0
65,Wilson Richards,10001,10001,11,0.023699,0.047397,2.0
68,Kate Winthrop,10004,10004,11,0.019910,0.039819,2.0
69,Hank Samson,10015,10015,11,0.016135,0.032271,2.0
67,Kōhaku Narukami,10012,10012,11,0.015631,0.031262,2.0


=== inv_cycle = 12 (pool: Decklist.cycle >= 12) ===


,investigator_name,canonical_front,canonical_back,Cycle,Popularity,investigator_weight,pool_weight
70,George Barnaby,11017,11017,12,0.063116,0.063116,1.0
73,Michael McGlen,11011,11011,12,0.062906,0.062906,1.0
72,Marion Tavares,11001,11001,12,0.055365,0.055365,1.0
76,Lucius Galloway,11004,11004,12,0.042816,0.042816,1.0
75,Gloria Goldberg,11014,11014,12,0.042170,0.042170,1.0
71,Agatha Crane,11008,11008,12,0.027683,0.027683,1.0
74,Agatha Crane,11007,11007,12,0.019488,0.019488,1.0


In [94]:
# Card popularity within a single investigator (spec P1–P5 + bias compensation)

DISPLAY_COLS = [
  'canonical_id', 'card_index', 'option_index', 'name',
  'cycle', 'xp', 'slot', 'P3', 'P4', 'Popularity',
]


def extract_investigator_cards(canonical_front: str,
                               canonical_back: str,
                               min_popularity: float = 0.0) -> pd.DataFrame:
  """Return spec P1–P5 popularity for a (canonical_front, canonical_back) tuple."""
  rows = pop_engine.popularity_for_investigator(
    prepared_decks,
    canonical_front,
    canonical_back,
  )
  df = pd.DataFrame(rows)
  if df.empty:
    return df
  df['cycle'] = df['canonical_id'].map(
    lambda cid: pop_engine.canonical_cards[cid].cycle
    if cid in pop_engine.canonical_cards else None
  )
  df = df[df['p5_popularity'] >= min_popularity]
  return df.rename(columns={
    'p3_opportunity_weight': 'P3',
    'p4_choice_weight': 'P4',
    'p5_popularity': 'Popularity',
  }).sort_values('Popularity', ascending=False)


def show_investigator_card_popularity(canonical_front: str,
                                      canonical_back: str,
                                      *,
                                      min_popularity: float = 0.0,
                                      head: int = 40) -> None:
  """Show separate 0 XP and 1+ XP popularity tables with CanonicalCard.cycle."""
  df = extract_investigator_cards(
    canonical_front, canonical_back, min_popularity=min_popularity,
  )
  inv_name = (
    df_decks.loc[
      (df_decks['canonical_front'] == canonical_front)
      & (df_decks['canonical_back'] == canonical_back),
      'investigator_name',
    ].iloc[0]
    if len(df_decks) else canonical_front
  )
  print(f'{inv_name} ({canonical_front} / {canonical_back})')

  zero_xp = df[df['xp'].fillna(0) == 0].copy()
  plus_xp = df[df['xp'].fillna(0) > 0].copy()

  # Ensure 'card_index', 'cycle', and 'xp' display as int where possible
  for table in [zero_xp, plus_xp]:
    if 'card_index' in table.columns:
      table['card_index'] = table['card_index'].fillna(0).astype(int)
    if 'cycle' in table.columns:
      table['cycle'] = table['cycle'].fillna(0).astype(int)
    if 'xp' in table.columns:
      table['xp'] = table['xp'].fillna(0).astype(int)

  print(f'\n--- 0 XP cards ({len(zero_xp)} options) ---')
  display(zero_xp[DISPLAY_COLS].head(head))

  print(f'\n--- 1+ XP cards ({len(plus_xp)} options) ---')
  display(plus_xp[DISPLAY_COLS].head(head))


# Example: Daisy Walker (03002)
show_investigator_card_popularity('03002', '03002')

Minh Thi Phan (03002 / 03002)

--- 0 XP cards (444 options) ---


,canonical_id,card_index,option_index,name,cycle,xp,slot,P3,P4,Popularity
0,03010,1,None,Analytical Mind,3,0,,4.853610,4.853610,1.000000
1,03011,1,None,The King in Yellow,3,0,Hand,4.853610,4.853610,1.000000
2,01039,1,None,Deduction,1,0,,7.114098,6.244839,0.879934
3,01039,2,None,Deduction,1,0,,7.114098,6.041152,0.843377
4,01000,1,None,Random Basic Weakness,1,0,,7.114098,5.574552,0.769941
5,01090,1,None,Perception,1,0,,7.114098,5.199367,0.728948
6,01090,2,None,Perception,1,0,,7.114098,4.717401,0.653023
7,02227,1,None,Inquiring Mind,2,0,,5.643071,3.314990,0.619542
8,03231,1,None,Eureka!,3,0,,4.853610,2.658952,0.569207
9,01093,1,None,Unexpected Courage,1,0,,7.114098,3.689054,0.565863



--- 1+ XP cards (260 options) ---


,canonical_id,card_index,option_index,name,cycle,xp,slot,P3,P4,Popularity
15,60228,1,None,Perception,7,2,None,3.525668,1.332512,0.406199
19,60228,2,None,Perception,7,2,None,3.525668,1.214153,0.374098
21,05194,1,None,Grisly Totem,5,3,Accessory,3.364388,1.119438,0.365503
29,02150,1,None,Deduction,2,2,,3.000205,0.690523,0.278259
32,11049,1,None,Antikythera,12,5,Accessory,4.056565,1.081081,0.266502
33,02150,2,None,Deduction,2,2,,3.000205,0.636403,0.253414
35,07196,1,None,Unrelenting,8,1,None,3.932138,0.788044,0.249358
36,10118,1,None,Persistence,11,1,None,4.962169,1.255869,0.248666
38,06204,1,None,Sharp Vision,6,1,,3.299938,0.605754,0.237169
39,06204,2,None,Sharp Vision,6,1,,3.299938,0.605754,0.237169


In [67]:
# Change canonical_front/back to explore another investigator.
show_investigator_card_popularity('01004', '01004', head=60)

Agnes Baker (01004 / 01004)

--- 0 XP cards (503 options) ---


,canonical_id,card_index,option_index,name,cycle,xp,slot,P3,P4,Popularity
0,01012,1,None,Heirloom of Hyperborea,1,0,Accessory,5.769118,5.668929,0.987211
1,01013,1,None,Dark Memory,1,0,,5.769118,5.668929,0.987211
2,01060,1,None,Shrivelling,1,0,Arcane,5.769118,4.509354,0.806855
3,01000,1,None,Random Basic Weakness,1,0,,5.769118,4.283011,0.736340
4,01060,2,None,Shrivelling,1,0,Arcane,5.769118,3.994957,0.709572
5,01065,1,None,Ward of Protection,1,0,,5.769118,3.803834,0.684487
6,01058,1,None,Forbidden Knowledge,1,0,,5.769118,3.763543,0.660046
7,01067,1,None,Fearless,1,0,,5.769118,3.406186,0.613131
8,01065,2,None,Ward of Protection,1,0,,5.769118,3.438452,0.609678
9,01059,1,None,Holy Rosary,1,0,Accessory,5.769118,3.262084,0.591363



--- 1+ XP cards (335 options) ---


,canonical_id,card_index,option_index,name,cycle,xp,slot,P3,P4,Popularity
28,02154,1,None,Shrivelling,2,3,Arcane,2.352079,0.584891,0.307532
31,11074,1,None,Sword Cane,12,2,Hand,4.296387,1.180374,0.274736
34,02035,1,None,Peter Sylvestre,2,2,Ally,2.352079,0.611894,0.264474
42,08090,1,None,Brand of Cthugha,9,1,Arcane,3.928642,0.836985,0.228220
45,02035,2,None,Peter Sylvestre,2,2,Ally,2.352079,0.520012,0.217836
48,02154,2,None,Shrivelling,2,3,Arcane,2.352079,0.409441,0.211960
58,11077,1,None,Enraptured,12,2,None,4.296387,0.728972,0.169671
59,11077,2,None,Enraptured,12,2,None,4.296387,0.728972,0.169671
61,02158,1,None,Charisma,2,3,,2.352079,0.360749,0.167138
63,11080,1,None,Eldritch Brand,12,10,None,4.296387,0.704799,0.164044
